# Heart Health Dataset: Data Cleaning & Preprocessing
## BRFSS 2015 Heart Disease Health Indicators

### Project Overview & Purpose
This notebook performs comprehensive data cleaning, categorical mapping, and feature engineering on the **CDC BRFSS 2015 Heart Disease Indicators** dataset (`RAW/heart_disease_health_indicators_BRFSS2015 2 (1).csv`). 

The objective is to prepare a clean, well-structured dataset for exploratory data analysis (EDA) and visualization, guided by the official data dictionary (`RAW/Health and demographic data description.pdf`).

---

### Step 1: Environment Setup & Library Imports

**Why this is being done (Logic):**
Before performing data cleaning operations, we must load the required Python software libraries. `pandas` provides powerful tabular data structures, `numpy` handles numerical operations, and `os` manages directory paths for saving outputs.

**How it is being done (Code Explanation):**
- `import pandas as pd`: Imports the Pandas library with alias `pd`.
- `import numpy as np`: Imports NumPy library for numerical arrays and calculations.
- `import os`: Imports OS library to create output folders and manage file paths.
- `import warnings`: Suppresses minor non-critical warnings for clean output.
- `pd.set_option(...)`: Configures Pandas display options to show all columns clearly without truncation.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

# Suppress non-critical warnings
warnings.filterwarnings('ignore')

# Set display options for clean output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment successfully setup with Pandas version:", pd.__version__)

Environment successfully setup with Pandas version: 2.2.3


### Step 2: Data Loading & Initial Inspection

**Why this is being done (Logic):**
To begin cleaning, we load the raw CSV file into memory and inspect its size, column structure, and sample data rows. This initial check verifies that the raw dataset is accessible and properly formatted.

**How it is being done (Code Explanation):**
- `os.path.join('RAW', ...)`: Constructs a cross-platform file path to the raw dataset.
- `pd.read_csv(...)`: Reads the raw CSV file into a Pandas DataFrame named `df_raw`.
- `df_raw.shape`: Displays dataset dimensions (total rows and total columns).
- `df_raw.head()`: Displays the first 5 rows of data for visual inspection.

In [2]:
# Define relative path to raw dataset
raw_csv_path = os.path.join('RAW', 'heart_disease_health_indicators_BRFSS2015 2 (1).csv')

# Read raw dataset
df_raw = pd.read_csv(raw_csv_path)

# Display shape and head
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
print("First 5 rows of raw dataset:")
df_raw.head()

Raw Dataset Shape: 253,680 rows and 22 columns
First 5 rows of raw dataset:


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40,1,0,0,0,0,1,0,1,0,5,18,15,1,0,9,4,3
1,0,0,0,0,25,1,0,0,1,0,0,0,0,1,3,0,0,0,0,7,6,1
2,0,1,1,1,28,0,0,0,0,1,0,0,1,1,5,30,30,1,0,9,4,8
3,0,1,0,1,27,0,0,0,1,1,1,0,1,0,2,0,0,0,0,11,3,6
4,0,1,1,1,24,0,0,0,1,1,1,0,1,0,2,3,0,0,0,11,5,4


### Step 3: Assessing Missing Values & Initial Data Types

**Why this is being done (Logic):**
Missing values (nulls) can cause calculation errors or bias during visualization. We must check every column for missing entries and inspect the raw data types assigned by Pandas to plan appropriate conversions.

**How it is being done (Code Explanation):**
- `df_raw.isnull().sum()`: Counts missing values (`NaN`/`None`) in each column.
- `df_raw.dtypes`: Lists the data type assigned to each column (e.g. `int64`, `float64`).
- We combine these into a summary table showing column name, data type, missing count, and missing percentage.

In [3]:
# Count missing values per column
missing_counts = df_raw.isnull().sum()

# Summary table for data types and missing values
data_summary = pd.DataFrame({
    'Data_Type': df_raw.dtypes,
    'Missing_Values': missing_counts,
    'Missing_Percentage (%)': (missing_counts / len(df_raw)) * 100
})

print("=== Raw Data Quality & Missing Value Audit ===")
data_summary

=== Raw Data Quality & Missing Value Audit ===


,Data_Type,Missing_Values,Missing_Percentage (%)
HeartDiseaseorAttack,int64,0,0.0
HighBP,int64,0,0.0
HighChol,int64,0,0.0
CholCheck,int64,0,0.0
BMI,int64,0,0.0
Smoker,int64,0,0.0
Stroke,int64,0,0.0
Diabetes,int64,0,0.0
PhysActivity,int64,0,0.0
Fruits,int64,0,0.0


### Step 4: Identifying & Handling Duplicate Rows

**Why this is being done (Logic):**
In population health survey datasets like BRFSS, identical survey answers can occur across multiple respondents due to discrete integer coding. However, keeping thousands of duplicate rows can artificially skew frequency counts and summary statistics during exploratory visualization. Removing exact duplicate rows ensures each row represents a unique response combination.

**How it is being done (Code Explanation):**
- `df_raw.duplicated().sum()`: Counts rows where every column value matches another row exactly.
- `df_raw.drop_duplicates()`: Removes exact duplicate rows and creates `df`.
- `df.reset_index(drop=True)`: Re-indexes the dataset sequentially starting from 0.

In [4]:
# Count duplicate rows
num_duplicates = df_raw.duplicated().sum()
print(f"Total duplicate rows detected: {num_duplicates:,}")

# Drop duplicate rows
df = df_raw.drop_duplicates().copy()

# Reset index sequentially
df = df.reset_index(drop=True)

print(f"Dataset shape after removing duplicates: {df.shape[0]:,} rows and {df.shape[1]} columns")

Total duplicate rows detected: 23,899
Dataset shape after removing duplicates: 229,781 rows and 22 columns


### Step 5: Converting Categorical Variables Using Data Dictionary

**Why this is being done (Logic):**
The raw dataset uses numeric codes for categorical features (e.g., `Sex`: 0/1, `Diabetes`: 0/1/2, `Age`: 1 to 13, `Income`: 1 to 8). Reading numeric codes in visualizations makes charts hard to interpret. We convert these codes into human-readable text categories using the official PDF data dictionary (`RAW/Health and demographic data description.pdf`).

**How it is being done (Code Explanation):**
- We create Python dictionary mappings corresponding to each variable's data dictionary description:
  - `Sex`: `0` -> `'Female'`, `1` -> `'Male'`
  - `Diabetes`: `0` -> `'No Diabetes'`, `1` -> `'Prediabetes'`, `2` -> `'Diabetes'`
  - `GenHlth`: `1` -> `'1 - Excellent'`, `2` -> `'2 - Very Good'`, `3` -> `'3 - Good'`, `4` -> `'4 - Fair'`, `5` -> `'5 - Poor'`
  - `Age`: Maps levels 1-13 to 5-year age intervals (`'18-24'`, `'25-29'`, ..., `'80+'`).
  - `Education`: Maps levels 1-6 to education attainment descriptions.
  - `Income`: Maps levels 1-8 to household income brackets.
  - Binary Indicators: Maps `0` -> `'No'` and `1` -> `'Yes'`.
- We use `.map()` to generate explicit text label columns alongside original numeric flags.

In [5]:
# 1. Sex Mapping (0 = Female, 1 = Male)
sex_map = {0: 'Female', 1: 'Male'}
df['Sex_Label'] = df['Sex'].map(sex_map)

# 2. Diabetes Status Mapping (0 = No Diabetes, 1 = Prediabetes, 2 = Diabetes)
diabetes_map = {
    0: 'No Diabetes',
    1: 'Prediabetes',
    2: 'Diabetes'
}
df['Diabetes_Status'] = df['Diabetes'].map(diabetes_map)

# 3. General Health Rating Mapping (1 = Excellent to 5 = Poor)
genhlth_map = {
    1: '1 - Excellent',
    2: '2 - Very Good',
    3: '3 - Good',
    4: '4 - Fair',
    5: '5 - Poor'
}
df['General_Health'] = df['GenHlth'].map(genhlth_map)

# 4. Age Category Mapping (13 levels in 5-year increments)
age_map = {
    1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39',
    5: '40-44', 6: '45-49', 7: '50-54', 8: '55-59',
    9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79',
    13: '80+'
}
df['Age_Category'] = df['Age'].map(age_map)

# 5. Education Level Mapping (1-6 scale)
education_map = {
    1: 'Never attended / Kindergarten',
    2: 'Grades 1-8 (Elementary)',
    3: 'Grades 9-11 (Some High School)',
    4: 'Grade 12 / GED (High School Grad)',
    5: 'College 1-3 yrs (Some College / Tech)',
    6: 'College 4+ yrs (College Grad)'
}
df['Education_Level'] = df['Education'].map(education_map)

# 6. Income Bracket Mapping (1-8 scale)
income_map = {
    1: 'Less than $10,000',
    2: '$10,000 to < $15,000',
    3: '$15,000 to < $20,000',
    4: '$20,000 to < $25,000',
    5: '$25,000 to < $35,000',
    6: '$35,000 to < $50,000',
    7: '$50,000 to < $75,000',
    8: '$75,000 or more'
}
df['Income_Bracket'] = df['Income'].map(income_map)

# 7. Binary Indicator Mapping (0 = No, 1 = Yes)
binary_map = {0: 'No', 1: 'Yes'}

binary_cols_to_map = {
    'HeartDiseaseorAttack': 'Heart_Disease_or_Attack',
    'HighBP': 'High_BP_Label',
    'HighChol': 'High_Cholesterol_Label',
    'CholCheck': 'Cholesterol_Check_5yrs',
    'Smoker': 'Smoker_Status',
    'Stroke': 'Stroke_History',
    'PhysActivity': 'Physical_Activity',
    'Fruits': 'Fruit_Consump_Daily',
    'Veggies': 'Veggie_Consump_Daily',
    'HvyAlcoholConsump': 'Heavy_Alcohol_Consump',
    'AnyHealthcare': 'Healthcare_Coverage',
    'NoDocbcCost': 'No_Doctor_Due_To_Cost',
    'DiffWalk': 'Difficulty_Walking'
}

for orig_col, new_label_col in binary_cols_to_map.items():
    df[new_label_col] = df[orig_col].map(binary_map)

print("Categorical mappings completed successfully!")
df[['Sex_Label', 'Diabetes_Status', 'General_Health', 'Age_Category', 'Income_Bracket']].head()

Categorical mappings completed successfully!


,Sex_Label,Diabetes_Status,General_Health,Age_Category,Income_Bracket
0,Female,No Diabetes,5 - Poor,60-64,"$15,000 to < $20,000"
1,Female,No Diabetes,3 - Good,50-54,"Less than $10,000"
2,Female,No Diabetes,5 - Poor,60-64,"$75,000 or more"
3,Female,No Diabetes,2 - Very Good,70-74,"$35,000 to < $50,000"
4,Female,No Diabetes,2 - Very Good,70-74,"$20,000 to < $25,000"


### Step 6: Reviewing & Removing Irrelevant Columns

**Why this is being done (Logic):**
Before proceeding to exploratory analysis, we review all variables to confirm their relevance. In this dataset, all 22 raw columns represent clinical factors, medical history, lifestyle behaviors, or demographic attributes essential for heart health research. Therefore, all raw columns are kept. We demonstrate simple Pandas code for dropping columns should any feature be deemed unwanted in future iterations.

**How it is being done (Code Explanation):**
- We inspect total current columns.
- We show `df.drop(columns=[...])` code with simple variable definitions.

In [6]:
# Column review: All 22 original survey indicators are clinically relevant.
columns_to_remove = [] # List any column names here to drop if needed

if len(columns_to_remove) > 0:
    df = df.drop(columns=columns_to_remove)
    print(f"Removed columns: {columns_to_remove}")
else:
    print("No columns removed. All original features retained alongside mapped categorical labels.")

No columns removed. All original features retained alongside mapped categorical labels.


### Step 7: Feature Engineering (Creating New Columns for Analysis Clarity)

**Why this is being done (Logic):**
Creating derived features simplifies comparative analysis and pattern discovery during EDA:
1. **`BMI_Category`**: Categorizes numerical BMI into WHO standard groups (`Underweight`, `Normal Weight`, `Overweight`, `Obese`).
2. **`Age_Group_Broad`**: Aggregates 13 narrow age brackets into 3 broad life stages (`Young Adults: 18-39`, `Middle-Aged Adults: 40-59`, `Older Adults: 60+`).
3. **`Total_Unhealthy_Days`**: Sums physical bad health days (`PhysHlth`) and mental bad health days (`MentHlth`) in the past 30 days.
4. **`Cardiovascular_Risk_Score`**: Combines 6 key heart disease risk factors (High BP, High Cholesterol, Smoking history, Diabetes, Physical Inactivity, Heavy Alcohol) into a composite risk score (0 to 6).

**How it is being done (Code Explanation):**
- `pd.cut(df['BMI'], ...)`: Bins continuous BMI numbers into WHO categories.
- `.apply(assign_broad_age_group)`: Applies a custom function to assign age codes to 3 broad age stages.
- `df['PhysHlth'] + df['MentHlth']`: Simple addition of unhealthy days.
- Additive sum of risk indicator flags to construct `Cardiovascular_Risk_Score`.

In [8]:
# 1. Feature 1: WHO BMI Categories
bmi_bins = [0, 18.5, 24.9, 29.9, 100]
bmi_labels = ['Underweight', 'Normal Weight', 'Overweight', 'Obese']
df['BMI_Category'] = pd.cut(df['BMI'], bins=bmi_bins, labels=bmi_labels, right=True)

# 2. Feature 2: Broad Age Group (18-39, 40-59, 60+)
def assign_broad_age_group(age_code):
    if age_code <= 4:
        return 'Young Adults (18-39)'
    elif age_code <= 8:
        return 'Middle-Aged Adults (40-59)'
    else:
        return 'Older Adults (60+)'

df['Age_Group_Broad'] = df['Age'].apply(assign_broad_age_group)

# 3. Feature 3: Total Unhealthy Days (Physical + Mental bad health days)
df['Total_Unhealthy_Days'] = df['PhysHlth'] + df['MentHlth']

# 4. Feature 4: Cardiovascular Risk Factor Score (0 to 6 scale)
# Risk components: High BP, High Cholesterol, Smoker, Diabetes, Inactivity (PhysActivity==0), Heavy Alcohol
df['Cardiovascular_Risk_Score'] = (
    df['HighBP'] + 
    df['HighChol'] + 
    df['Smoker'] + 
    (df['Diabetes'] > 0).astype(int) + 
    (df['PhysActivity'] == 0).astype(int) + 
    df['HvyAlcoholConsump']
)

print("Feature engineering completed!")
df[['BMI', 'BMI_Category', 'Age_Category', 'Age_Group_Broad', 'PhysHlth', 'MentHlth', 'Total_Unhealthy_Days', 'Cardiovascular_Risk_Score']].head()

Feature engineering completed!


,BMI,BMI_Category,Age_Category,Age_Group_Broad,PhysHlth,MentHlth,Total_Unhealthy_Days,Cardiovascular_Risk_Score
0,40,Obese,60-64,Older Adults (60+),15,18,33,4
1,25,Overweight,50-54,Middle-Aged Adults (40-59),0,0,0,1
2,28,Overweight,60-64,Older Adults (60+),30,30,60,3
3,27,Overweight,70-74,Older Adults (60+),0,0,0,1
4,24,Normal Weight,70-74,Older Adults (60+),0,3,3,2


### Step 8: Final Cleanliness Audit & Data Verification

**Why this is being done (Logic):**
Before exporting, we conduct a final data verification to guarantee:
1. Zero missing or NaN values exist.
2. Dataset dimensions match expectations after deduplication and feature addition.
3. Target variable distribution (`HeartDiseaseorAttack`) is documented.

**How it is being done (Code Explanation):**
- `df.isnull().sum().sum()`: Sums all missing values across the entire DataFrame.
- `df['HeartDiseaseorAttack'].value_counts(...)`: Calculates counts and percentages for Heart Disease status.

In [9]:
# Check total missing values
total_missing = df.isnull().sum().sum()
print(f"Final Missing Values Count: {total_missing}")

# Check dataset dimensions
print(f"Final Cleaned Dataset Shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

# Target variable class balance summary
target_counts = df['HeartDiseaseorAttack'].value_counts()
target_pcts = (df['HeartDiseaseorAttack'].value_counts(normalize=True) * 100).round(2)

target_summary = pd.DataFrame({
    'Record_Count': target_counts,
    'Percentage (%)': target_pcts
})
target_summary.index = ['No Heart Disease / Attack (0)', 'Heart Disease / Attack (1)']

print("Target Variable Distribution:")
target_summary

Final Missing Values Count: 0
Final Cleaned Dataset Shape: 229,781 rows and 45 columns
Target Variable Distribution:


,Record_Count,Percentage (%)
No Heart Disease / Attack (0),206064,89.68
Heart Disease / Attack (1),23717,10.32


### Step 9: Exporting the Cleaned Dataset

**Why this is being done (Logic):**
To ensure clean separation between raw and processed data, we save the cleaned DataFrame into a dedicated `CLEANED` folder as `heart_disease_cleaned.csv`.

**How it is being done (Code Explanation):**
- `os.makedirs('CLEANED', exist_ok=True)`: Creates the destination directory if it does not exist.
- `df.to_csv('CLEANED/heart_disease_cleaned.csv', index=False)`: Writes the clean DataFrame to disk without adding extra row index columns.

In [10]:
# Create CLEANED folder
output_dir = 'CLEANED'
os.makedirs(output_dir, exist_ok=True)

# Export cleaned dataframe
output_csv_path = os.path.join(output_dir, 'heart_disease_cleaned.csv')
df.to_csv(output_csv_path, index=False)

print(f"Cleaned dataset successfully exported to: {output_csv_path}")
print(f"File size: {os.path.getsize(output_csv_path) / (1024 * 1024):.2f} MB")

Cleaned dataset successfully exported to: CLEANED\heart_disease_cleaned.csv
File size: 47.66 MB


### Step 10: Cleaning Summary & Dataset Status

#### Summary of Completed Actions:
- **Raw Data Ingestion**: Audited 253,680 survey records across 22 health indicator features.
- **Duplicate Removal**: Handled 23,899 duplicate records, yielding **229,781 unique respondent records**.
- **Categorical Transformation**: Mapped numeric codes for `Sex`, `Diabetes`, `GenHlth`, `Age`, `Education`, `Income`, and all binary indicators to descriptive human-readable text strings based on `Health and demographic data description.pdf`.
- **Feature Engineering**: Created 4 new analytical features (`BMI_Category`, `Age_Group_Broad`, `Total_Unhealthy_Days`, `Cardiovascular_Risk_Score`).
- **Data Integrity Audit**: Confirmed 0 missing values across all 43 columns (22 original + 17 categorical label columns + 4 derived features).
- **Export**: Dataset saved to `CLEANED/heart_disease_cleaned.csv`, ready for EDA and visual analysis.